In [5]:
# 1. wconcept 크롤링 준비
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

driver.quit()



In [8]:
# 2. wconcept 베스트 카테고리 접속
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수", len(items))

driver.quit()



찾은 상품 수 200


In [20]:
# 3. wconcept 베스트 카테고리 접속 > 상세정보 수집

# 판매가 기준 상위 20개 상품 조회
# 할인율이 30%이상인 상품 조회 & 내림차순 정렬
# 특정 브랜드 상품만 조회, 판매가 기준 오름차순 정렬
# 리뷰수가 100이상인 상품의 평균 판매가 조회
# 브랜드별 상품수 & 평균할인율 구하고 상품수가 많은 브랜드 10개 조회

# 소비자가 / 판매가 / 할인률 / 브랜드 / 상품명 / 리뷰수 / 좋아요갯수

# 네이밍정보 1개 -> (브랜드명 + 상품명)
# 가격정보 1개 -> (소비자가 + 할인율 + 판매가)
# 서브정보 1개 -> (평점 + 리뷰갯수 + 좋아요갯수)

# 테이블 나누는 기준 : 식별성을 가지고 있어야 하는 대상이 어떤 것인가?
# sakila : 렌탈 / 카테고리 / 영화

# 브랜드 전용 테이블
# 상품관련 테이블

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num)

for idx, item in enumerate(items, 1) :
    # 브랜드 & 상품정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

    # 소비자가 & 판매가 & 할인률정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try :
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except :
        original_price = 0

    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except :
        sale_price = 0

    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except :
        discount_rate = 0
        
    # 평점 & 리뷰수 & 좋아요
    stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

    if stats :
        stats = stats[0]
    
        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
        except :
            rating = 0
    
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
        except :
            review_count = 0
    
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
        except :
            like_count = 0

    else :
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품링크
    img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")

    match = re.search(r"/(\d{9})_", img_src)

    if match :
        product_no = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
    else :
        product_no = None
        product_url = ""
    
    print(idx, brand_name, product_name, original_price, sale_price, discount_rate, rating, review_count, like_count, product_url)
    # print(idx, brand_name, product_name, original_price, sale_price, discount_rate, rating, review_count, like_count, product_url)

driver.quit()



찾은 상품 수 200
1 프론트로우 [이태리 트위드]Trimmed Over-fit Tweed Jacket_2color 279000 208692 25 5.0 18 2667 https://www.wconcept.co.kr/Product/308023502
2 프론트로우 [Drama Signature] Relaxed Bootcut Trousers_4color 149000 131120 12 4.9 288 6462 https://www.wconcept.co.kr/Product/305613843
3 프론트로우 Mid-rise Bootscut Denim Pants_3color 99000 78408 20 5.0 123 2973 https://www.wconcept.co.kr/Product/306581931
4 프론트로우 [이태리울]Over-fit Wool Single Jacket_3color 259000 205128 20 5.0 6 1228 https://www.wconcept.co.kr/Product/308015094
5 마른파이브 [26SS]쉬어 스칼렛 시스루 레이스 브라렛 팬티 세트 59800 36900 38 0 0 46 https://www.wconcept.co.kr/Product/308296933
6 하시에 [한정특가] (6차 리오더) CASHMERE COLLAR LIGHT DOWN JACKET [IVORY][BLACK] 178000 117480 34 4.9 702 9999 https://www.wconcept.co.kr/Product/303596201
7 프론트로우 [Drama Signature] Tie-Collar Silky Blouse_6color 149000 118008 20 4.9 1444 9999 https://www.wconcept.co.kr/Product/301827414
8 브아빗포우먼 [한정특가] [세트]버튼 스트라이프 싱글 자켓+맥시 스트라이프 스커트 네이비 VW711 328000 233077 28 5.0 5 1000 https://www.wcon

In [22]:
# 3. wconcept 베스트 카테고리 접속 > 상세정보 수집 > MySQL 데이터 전송

# 판매가 기준 상위 20개 상품 조회
# 할인율이 30%이상인 상품 조회 & 내림차순 정렬
# 특정 브랜드 상품만 조회, 판매가 기준 오름차순 정렬
# 리뷰수가 100이상인 상품의 평균 판매가 조회
# 브랜드별 상품수 & 평균할인율 구하고 상품수가 많은 브랜드 10개 조회

# 소비자가 / 판매가 / 할인률 / 브랜드 / 상품명 / 리뷰수 / 좋아요갯수

# 네이밍정보 1개 -> (브랜드명 + 상품명)
# 가격정보 1개 -> (소비자가 + 할인율 + 판매가)
# 서브정보 1개 -> (평점 + 리뷰갯수 + 좋아요갯수)

# 테이블 나누는 기준 : 식별성을 가지고 있어야 하는 대상이 어떤 것인가?
# sakila : 렌탈 / 카테고리 / 영화

# 브랜드 전용 테이블
# 상품관련 테이블

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

# mysql 접속정보
conn = pymysql.connect(
    host = "127.0.0.1",
    user = "root",
    password = "126845as",
    db = "wconcept_db",
    charset = "utf8mb4"
)

cursor = conn.cursor()

# 가상 브라우저
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num)

for idx, item in enumerate(items, 1) :
    # 브랜드 & 상품정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

    # 소비자가 & 판매가 & 할인률정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try :
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except :
        original_price = 0

    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except :
        sale_price = 0

    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except :
        discount_rate = 0
        
    # 평점 & 리뷰수 & 좋아요
    stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

    if stats :
        stats = stats[0]
    
        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
        except :
            rating = 0
    
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
        except :
            review_count = 0
    
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
        except :
            like_count = 0

    else :
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품링크
    img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")

    match = re.search(r"/(\d{9})_", img_src)

    if match :
        product_no = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
    else :
        product_no = None
        product_url = ""

    # mysql 테이블 저장 - brands
    cursor.execute("""
        INSERT INTO brands (brand_name) VALUES (%s) ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
    """, (brand_name,))

    cursor.execute("SELECT brand_id FROM brands WHERE brand_name = %s", (brand_name))
    brand_id = cursor.fetchone()[0]
    
    # mysql 테이블 저장 - products
    cursor.execute("""
        INSERT INTO products (brand_id, rank_no, product_name, orginal_price,
        sale_price, discount_rate, rating, review_count, like_count, product_url
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (brand_id, idx, product_name, sale_price, discount_rate, rating, review_count,
          like_count, product_url))

    print(f"{idx}위 저장 완료: {brand_name} / {product_name} / {product_no}")

conn.comit()
conn.close()

driver.quit()

찾은 상품 수 200


TypeError: not enough arguments for format string

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

# mysql 접속정보
conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password="126845as",
    db="wconcept_db",
    charset="utf8mb4"
)

cursor = conn.cursor()

# 가상 브라우저
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_btn.click()
time.sleep(3)

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수", len(items))

def to_int(text):
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

for idx, item in enumerate(items, 1):
    # 브랜드 & 상품정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

    # 소비자가 & 판매가 & 할인률정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try:
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except:
        original_price = 0

    try:
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except:
        sale_price = 0

    try:
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except:
        discount_rate = 0

    # 평점 & 리뷰수 & 좋아요
    stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

    if stats:
        stats = stats[0]

        try:
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
        except:
            rating = 0.0

        try:
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
        except:
            review_count = 0

        try:
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
        except:
            like_count = 0
    else:
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품링크
    img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")

    match = re.search(r"/(\d{9})_", img_src)

    if match:
        product_no = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
    else:
        product_no = None
        product_url = ""

    # mysql 테이블 저장 - brands
    cursor.execute("""
        INSERT INTO brands (brand_name)
        VALUES (%s)
        ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
    """, (brand_name,))

    cursor.execute("SELECT brand_id FROM brands WHERE brand_name = %s", (brand_name,))
    brand_id = cursor.fetchone()[0]

    # mysql 테이블 저장 - products
    cursor.execute("""
        INSERT INTO products (
            brand_id, rank_no, product_name, original_price,
            sale_price, discount_rate, rating, review_count,
            like_count, product_url
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        brand_id, idx, product_name, original_price,
        sale_price, discount_rate, rating, review_count,
        like_count, product_url
    ))

    print(f"{idx}위 저장 완료: {brand_name} / {product_name} / {product_no}")

conn.commit()
conn.close()
driver.quit()

찾은 상품 수 200
1위 저장 완료: 프론트로우 / [이태리 트위드]Trimmed Over-fit Tweed Jacket_2color / 308023502
2위 저장 완료: 프론트로우 / [Drama Signature] Relaxed Bootcut Trousers_4color / 305613843
3위 저장 완료: 프론트로우 / Mid-rise Bootscut Denim Pants_3color / 306581931
4위 저장 완료: 어스 / [한정특가] [7차리오더][기장선택가능] US102 레귤러 플레어진 (연청) / 307187866
5위 저장 완료: 룩캐스트 / [한정특가] [단독]루시 비건 레더 자켓_블랙 / LUCY VEGAN LEATHER JACKET_BLACK / 307368672
6위 저장 완료: 브아빗포우먼 / [한정특가] [세트]버튼 스트라이프 싱글 자켓+맥시 스트라이프 스커트 네이비 VW711 / 308295448
7위 저장 완료: 하시에 / [한정특가] (6차 리오더) CASHMERE COLLAR LIGHT DOWN JACKET [IVORY][BLACK] / 303596201
8위 저장 완료: 프론트로우 / [이태리울]Over-fit Wool Single Jacket_3color / 308015094
9위 저장 완료: 슬로우롤리 / [PRE-OPEN] [3차리오더] Denim washed work jacket - Light blue / 308317385
10위 저장 완료: 오브오브 / [한정특가] [단독] String Eco Leather Jacket_Black / 306631596
11위 저장 완료: 르하스 / TWO-WAY BUTTON JACKET_MELANGE BROWN / 307497276
12위 저장 완료: 마른파이브 / [26SS]쉬어 스칼렛 시스루 레이스 브라렛 팬티 세트 / 308296933
13위 저장 완료: 로브로브 / [그레이 W컨셉 단독][17차 리오더] 마이 애티튜드 니트 가디건_(8color) / 30605265

In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

# mysql 접속정보
conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password="126845as",
    db="wconcept_db",
    charset="utf8mb4"
)

cursor = conn.cursor()

# 가상 브라우저
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

driver.get(target_url)
time.sleep(3)

def to_int(text):
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

# 카테고리 버튼 클릭 함수
def click_category(category_name):
    category_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, f"//span[text()='{category_name}']")
        )
    )
    driver.execute_script("arguments[0].click();", category_btn)
    time.sleep(3)

# 현재 화면의 상품들 수집하는 함수
def collect_items(category_name):
    items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
    print(f"[{category_name}] 찾은 상품 수:", len(items))

    for idx, item in enumerate(items, 1):
        try:
            # 브랜드 & 상품정보
            brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
            brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
            product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()

            # 소비자가 & 판매가 & 할인률정보
            price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

            try:
                original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
            except:
                original_price = 0

            try:
                sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
            except:
                sale_price = 0

            try:
                discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
            except:
                discount_rate = 0

            # 평점 & 리뷰수 & 좋아요
            stats = item.find_elements(By.CSS_SELECTOR, "span.stats")

            if stats:
                stats = stats[0]

                try:
                    rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text.strip())
                except:
                    rating = 0.0

                try:
                    review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text.strip())
                except:
                    review_count = 0

                try:
                    like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like").text.strip())
                except:
                    like_count = 0
            else:
                rating = 0.0
                review_count = 0
                like_count = 0

            # 상품링크
            img_src = item.find_element(By.CSS_SELECTOR, "button.area-img img").get_attribute("src")
            match = re.search(r"/(\d{9})_", img_src)

            if match:
                product_no = match.group(1)
                product_url = f"https://www.wconcept.co.kr/Product/{product_no}"
            else:
                product_no = None
                product_url = ""

            # brands 저장
            cursor.execute("""
                INSERT INTO brands (brand_name)
                VALUES (%s)
                ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
            """, (brand_name,))

            cursor.execute("SELECT brand_id FROM brands WHERE brand_name = %s", (brand_name,))
            brand_id = cursor.fetchone()[0]

            # products 저장
            cursor.execute("""
                INSERT INTO products (
                    brand_id, category_name, rank_no, product_name, original_price,
                    sale_price, discount_rate, rating, review_count,
                    like_count, product_url
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (
                brand_id, category_name, idx, product_name, original_price,
                sale_price, discount_rate, rating, review_count,
                like_count, product_url
            ))

            print(f"[{category_name}] {idx}위 저장 완료: {brand_name} / {product_name} / {product_no}")

        except Exception as e:
            print(f"[{category_name}] {idx}번 상품 오류:", e)

# 먼저 베스트 버튼 클릭
best_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='베스트']")))
driver.execute_script("arguments[0].click();", best_btn)
time.sleep(3)

# 반복할 카테고리 목록
categories = ["의류", "가방", "신발", "액세서리"]

# 카테고리별 반복 수집
for category in categories:
    click_category(category)
    collect_items(category)
    conn.commit()

conn.close()
driver.quit()

[의류] 찾은 상품 수: 200
[의류] 1번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 2번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 3번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 4번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 5번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 6번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 7번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 8번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 9번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 10번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 11번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 12번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 13번 상품 오류: (1054, "Unknown column 'category_name' in 'field list'")
[의류] 14번 상품 오류: (1054, "Unknown column 'ca

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=145.0.7632.160)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x6e7dd3
	0x6e7e14
	0x4f1db0
	0x4d08d3
	0x5650fb
	0x57b0d9
	0x55e7d6
	0x530049
	0x530e04
	0x946924
	0x941bf7
	0x95f5a0
	0x700f58
	0x70891d
	0x6f0648
	0x6f0812
	0x6da21a
	0x764ffcc9
	0x773282ae
	0x7732827e
